In [6]:
import pandas as pd 
import numpy as np
import joblib
import shap
from sklearn.ensemble import GradientBoostingClassifier

/home/aximsoft/snap/code/261/.local/share/virtualenvs/week_10-6xdFAfPi/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
model = joblib.load(r"../models/model.pkl")
top_15_indices = joblib.load(r"../models/top_15_indices.pkl")
top_15_feature = joblib.load(r"../models/top_15_features.pkl")
x_train = np.load(r"../data/processed/x_train_fea.npy")
x_test = np.load(r"../data/processed/x_test_fea.npy")
y_train = np.load(r"../data/processed/y_train_fea.npy")
y_test = np.load(r"../data/processed/y_test_fea.npy")

In [7]:
x_train = x_train[:, top_15_indices]
x_test = x_test[:, top_15_indices]

In [10]:
explainer = shap.TreeExplainer(model)

In [11]:
values = explainer.shap_values(x_test)
values.shape

(1409, 15)

In [12]:
values

array([[-0.94208029, -0.46263113, -1.26198405, ..., -0.04502201,
        -0.09450047, -0.05644091],
       [ 0.54775693, -0.35837862,  0.56941322, ..., -0.01403319,
         0.04001019,  0.06384759],
       [-0.83529784, -0.3094299 , -0.06493647, ...,  0.00800032,
        -0.04160018, -0.03432146],
       ...,
       [ 0.46971941, -0.47094644,  0.30922729, ..., -0.00547159,
        -0.01874358,  0.02302264],
       [-0.83285866, -0.330559  , -0.25759138, ..., -0.0141894 ,
        -0.02166564,  0.03761394],
       [-0.90464965, -0.40781661, -1.26652702, ..., -0.01330037,
         0.01262859, -0.04665362]], shape=(1409, 15))

global importance

In [ ]:
global_importance = np.abs(values).mean(axis=0)

global_shap = sorted(zip(top_15_feature, global_importance),key=lambda x: x[1],reverse=True)

for feature, value in global_shap:
    print(feature, ":", value)

contract_risk : 0.6738669362512305
tenure : 0.3983309992526997
service_combination_risk : 0.35721506130313646
MonthlyCharges : 0.20739154801677645
InternetService_Fiber optic : 0.18834196025087685
Contract_Two year : 0.17085928546507761
charge_to_tenure_ratio : 0.16737568771488182
TotalCharges : 0.1545403899640244
payment_method_risk : 0.12962897066450027
PaperlessBilling_Yes : 0.12355891529768735
StreamingMovies_Yes : 0.06965106956070131
PaymentMethod_Electronic check : 0.0677585148977792
OnlineBackup_Yes : 0.044321091270692635
Dependents_Yes : 0.02885880231505273
Contract_One year : 0.01587435069247032


Individual customer contributor

In [16]:
cust_index = 0
for feature, value in zip(top_15_feature,values[0]):
    print(feature ,":", value)

contract_risk : -0.9420802867843566
service_combination_risk : -0.4626311289677882
tenure : -1.2619840537342182
InternetService_Fiber optic : 0.0888543167244651
MonthlyCharges : -0.24862020127628107
TotalCharges : 0.27753173975407264
charge_to_tenure_ratio : 0.20487554169151206
payment_method_risk : 0.01783086131326219
Contract_Two year : -0.7794219039404363
PaperlessBilling_Yes : 0.09182336155057709
PaymentMethod_Electronic check : -0.01774590439642987
StreamingMovies_Yes : 0.10188246069402342
Contract_One year : -0.04502200919109483
Dependents_Yes : -0.09450047249680013
OnlineBackup_Yes : -0.056440907815946566


positive contributor

In [29]:
customer_shap = values[0]

positive = []
negative = []

for feature, value in zip(top_15_feature, customer_shap):
    if value > 0:
        positive.append((feature, value))
    elif value < 0:
        negative.append((feature, value))

print("Positive churn contributors:")
for feature, value in sorted(positive, key=lambda x: x[1]):
    print(feature, ":", value)
print("*"*50)
print("Negative churn contributors:")
for feature, value in sorted(negative, key=lambda x: x[1]):
    print(feature, ":", value)

Positive churn contributors:
payment_method_risk : 0.01783086131326219
InternetService_Fiber optic : 0.0888543167244651
PaperlessBilling_Yes : 0.09182336155057709
StreamingMovies_Yes : 0.10188246069402342
charge_to_tenure_ratio : 0.20487554169151206
TotalCharges : 0.27753173975407264
**************************************************
Negative churn contributors:
tenure : -1.2619840537342182
contract_risk : -0.9420802867843566
Contract_Two year : -0.7794219039404363
service_combination_risk : -0.4626311289677882
MonthlyCharges : -0.24862020127628107
Dependents_Yes : -0.09450047249680013
OnlineBackup_Yes : -0.056440907815946566
Contract_One year : -0.04502200919109483
PaymentMethod_Electronic check : -0.01774590439642987
